[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lslumass/HyresBuilder/blob/dev/HyRes_Quickstart.ipynb)

# HyRes Coarse-Grained Protein Simulation — Quick-Start (Google Colab)

This notebook lets you build a **HyRes** coarse-grained (CG) model of a disordered protein or peptide directly from its amino acid sequence, and run a short OpenMM molecular dynamics simulation on a free Colab GPU — no local installation required.

It uses:
- **[HyresBuilder](https://github.com/lslumass/HyresBuilder)** — builds HyRes/iConRNA structures and PSF topology files (ChenLab, UMass Amherst)
- **[OpenMM](https://openmm.org/)** — runs the molecular dynamics simulation

**Relevant papers:**
1. Liu & Chen, *Phys. Chem. Chem. Phys.*, 2017, 19, 32421 — original HyRes model
2. Zhang, Liu & Chen, *J. Chem. Inf. Model.*, 2022, 62, 4523 — HyRes-II
3. Zhang, Li, Gong & Chen, *J. Am. Chem. Soc.*, 2024, 146, 342 — HyRes-GPU
4. Li & Chen, *PNAS*, 2025, 122, e2504583122

**Before you start:** go to `Runtime → Change runtime type` and select a **GPU** (e.g. T4). This notebook is also configured to request a GPU runtime automatically when opened.

---
### What this notebook does
1. Sets up a self-contained conda environment (needed because one dependency, `psfgen`, is conda-only) with OpenMM (CUDA-enabled) and HyresBuilder.
2. Builds a HyRes coarse-grained structure from a sequence you provide.
3. Generates the matching PSF topology file.
4. Runs a short HyRes MD simulation on the GPU.
5. Visualizes the trajectory and lets you download the results.


## 0. Check the GPU runtime

In [ ]:
!nvidia-smi

## 1. Install the environment

`psfgen` (a HyresBuilder dependency) is only distributed via `conda-forge`, so we install a lightweight Miniforge (conda-forge-only Miniconda) distribution inside the Colab VM and create a dedicated `hyres` environment there. This does **not** touch Colab's system Python — we call into the `hyres` env with `conda run` from here on, so no runtime restart is needed.

This step takes a few minutes; you only need to run it once per Colab session.

In [ ]:
%%bash
set -e
if [ ! -d /opt/conda ]; then
  echo "Downloading Miniforge..."
  wget -q https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh -O /tmp/miniforge.sh
  bash /tmp/miniforge.sh -b -p /opt/conda
fi
export PATH=/opt/conda/bin:$PATH

# Miniforge defaults to the conda-forge channel only (no Anaconda ToS wall)
if ! conda env list | grep -q "^hyres "; then
  conda create -n hyres -y python=3.9
fi
echo "Done."


In [ ]:
%%bash
set -e
export PATH=/opt/conda/bin:$PATH
source /opt/conda/etc/profile.d/conda.sh
conda activate hyres

# psfgen is conda-forge only
conda install -y -c conda-forge psfgen numpy numba mdanalysis

# OpenMM with the CUDA platform (pip wheels bundle CUDA 12 support directly)
pip install -q "openmm[cuda12]"

# HyresBuilder itself, straight from GitHub
pip install -q "git+https://github.com/lslumass/HyresBuilder.git"

echo "----- OpenMM installation test -----"
python -m openmm.testInstallation


In [ ]:
%%bash
# Also grab a local clone of the repo — we need scripts/run_latest.py,
# which is not installed as part of the pip package.
if [ ! -d HyresBuilder_repo ]; then
  git clone -q https://github.com/lslumass/HyresBuilder.git HyresBuilder_repo
fi
ls HyresBuilder_repo/scripts


## 2. Define your peptide

Edit `NAME` and `SEQUENCE` below. `SEQUENCE` should be a single-letter amino acid code with no spaces (e.g. `"GSMASNNTASIAQARKLVEQLKMEANIDRIKVSKAAADLMAYCEAHAKEDPLLTPVPASENPFREKKFFCAIL"`).

The default below is a short disordered test peptide so you can confirm the whole pipeline runs end-to-end quickly; replace it with your own sequence(s) of interest.

In [ ]:
NAME = "demo"
SEQUENCE = "GSMASNNTASIAQARKLVEQLKMEANIDRIKVSKAAADLMAYCEAHAKEDPLLTPVPASENPFREKKFFCAIL"

import os
os.makedirs("run", exist_ok=True)
with open("run/build_structure.py", "w") as f:
    f.write(f'''
from HyresBuilder import PeptideBuilder
PeptideBuilder.build_peptide("{NAME}", "{SEQUENCE}")
print("Built {NAME}.pdb")
''')
print("Wrote run/build_structure.py for", NAME)


In [ ]:
%%bash
export PATH=/opt/conda/bin:$PATH
source /opt/conda/etc/profile.d/conda.sh
conda activate hyres
cd run
python build_structure.py
ls -la


## 3. Generate the PSF topology file

`GenPsf.genpsf` uses `psfgen` under the hood to build the topology file HyRes/OpenMM needs alongside the PDB.

`terminal` controls the charge state of the N/C termini (`"neutral"`, `"charged"`, `"NT"`, or `"CT"`) — use whichever matches your experimental construct.

In [ ]:
TERMINAL = "neutral"

with open("run/build_psf.py", "w") as f:
    f.write(f'''
from HyresBuilder import GenPsf
GenPsf.genpsf("{NAME}.pdb", "{NAME}.psf", terminal="{TERMINAL}")
print("Built {NAME}.psf")
''')
print("Wrote run/build_psf.py")


In [ ]:
%%bash
export PATH=/opt/conda/bin:$PATH
source /opt/conda/etc/profile.d/conda.sh
conda activate hyres
cd run
python build_psf.py
ls -la


## 4. Run a short HyRes MD simulation

We use HyresBuilder's own `run_latest.py`, exactly as documented in the [HyresBuilder README](https://github.com/lslumass/HyresBuilder), rather than reimplementing the force field ourselves — this guarantees the simulation matches the published, benchmarked HyRes setup.

The example below runs a single chain in a **non-periodic** box (`-e non`), appropriate for a quick single-IDP test — the typical case for method demonstrations. For phase-separation (LLPS) or explicit-box simulations, switch to `-e NVT`/`-e NPT` and provide `-b` (box dimensions in nm), following the usage notes in the README.

**Note on run length:** `run_latest.py` is a well-tested production script with its simulation length/reporting set inside the file. For a quick demo run, or to match your paper's production parameters, open `run/HyresBuilder_repo/scripts/run_latest.py` in the Colab file browser (left sidebar) and adjust the step count / reporting interval directly, rather than guessing at undocumented flags here.

In [ ]:
TEMP = 298   # Kelvin
SALT = 150   # mM NaCl
OUT  = "demo_run"

cmd = (
    f"python HyresBuilder_repo/scripts/run_latest.py "
    f"-c run/{NAME}.pdb -p run/{NAME}.psf -o run/{OUT} "
    f"-t {TEMP} -e non -s {SALT}"
)
print(cmd)


In [ ]:
%%bash
export PATH=/opt/conda/bin:$PATH
source /opt/conda/etc/profile.d/conda.sh
conda activate hyres
python HyresBuilder_repo/scripts/run_latest.py -c run/demo.pdb -p run/demo.psf -o run/demo_run -t 298 -e non -s 150
ls -la run


## 5. Visualize the trajectory

`run_latest.py` writes an `.xtc` trajectory, a `.pdb`/log, and a checkpoint file with the prefix you gave it. We load the trajectory with MDAnalysis (in Colab's regular Python, not the `hyres` env) and render it with `py3Dmol`.

In [ ]:
!pip install -q py3Dmol MDAnalysis
import MDAnalysis as mda
import py3Dmol, glob

pdb_files = glob.glob("run/demo_run*.pdb") + ["run/demo.pdb"]
xtc_files = glob.glob("run/demo_run*.xtc")
print("PDB candidates:", pdb_files)
print("XTC candidates:", xtc_files)


In [ ]:
# Render the last frame of the trajectory (falls back to the built structure if no trajectory yet)
import glob

struct = glob.glob("run/demo_run*.pdb")
struct = struct[0] if struct else "run/demo.pdb"
traj = glob.glob("run/demo_run*.xtc")

if traj:
    u = mda.Universe(struct, traj[0])
    u.trajectory[-1]
    u.atoms.write("run/last_frame.pdb")
    view_file = "run/last_frame.pdb"
else:
    view_file = struct

with open(view_file) as f:
    pdb_block = f.read()

view = py3Dmol.view(width=600, height=500)
view.addModel(pdb_block, "pdb")
view.setStyle({"stick": {"radius": 0.3}, "sphere": {"scale": 0.3}})
view.zoomTo()
view.show()


## 6. Download your results

Zips up everything in `run/` (structure, PSF, trajectory, logs, checkpoint) so you can save it locally or attach it to your paper's supplementary data.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive("hyres_results", "zip", "run")
files.download("hyres_results.zip")


---
## Notes for experimentalists running this notebook

- Runtime disconnects after ~90 min idle on the free tier — re-run cells 1–1b to reinstall the environment if your session resets (Colab VMs are ephemeral; nothing above cell 6 persists between sessions).
- Colab's free GPU (T4) is a shared, non-guaranteed resource — for long production runs, use a paid Colab tier, a lab workstation, or an HPC cluster with the same `HyresBuilder` install steps (skip the Miniconda step and use your existing conda/mamba).
- For folded proteins / multi-chain or phase-separation (LLPS) systems, see the `at2hyres` and `examples/packmol.inp` instructions in the [HyresBuilder repo](https://github.com/lslumass/HyresBuilder) — the sequence-based workflow here is for single IDP chains.
- Questions about the model itself: contact the ChenLab (jianhanc@umass.edu).
